# Week 2b.2 — Agent Basics in LangChain

Now that we've covered the API basics in LangChain, we're ready to implement our first agent.

In [ ]:
from dotenv import load_dotenv
import os

load_dotenv()
assert os.getenv("GOOGLE_API_KEY"), "No GOOGLE_API_KEY found."
print("API key loaded")

## 0. Model Setup

Execute one of the following to define the model.

If using Ollama, you will need to start it first (simply open the chat UI and send a message):
- https://docs.langchain.com/oss/python/integrations/chat/ollama

Verify that Ollama is serving a model:
- http://localhost:11434/

In [ ]:
from langchain_ollama import ChatOllama

model = ChatOllama(model="qwen3.5:4b", reasoning=False)

In [ ]:
from langchain.chat_models import init_chat_model

model = init_chat_model(model="gpt-4.1-mini")

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI

model = ChatGoogleGenerativeAI(model="gemini-3.5-flash-lite")


## 1. The problem

LLMs can serve as the brain but their knowledge and reach are limited.

In [ ]:
from pprint import pprint

response = model.invoke("What time is it right now?")
pprint(response.content)

The model has no clock. Its weights were frozen months ago and nothing in the context tells it the time. It can only decline or guess.

We can surpass this limitations by augmenting the LLM with a tool.

## 2. Defining a tool

A tool starts life as an ordinary Python function.

In [ ]:
from datetime import datetime

def get_current_time():
    """Return the current local date and time."""
    return datetime.now().strftime("%A, %B %d, %Y at %I:%M %p")

In [ ]:
get_current_time()

### Requirements for Tool Definition
To make this usable by a model, LangChain needs three things: 
- a name (start with a verb and describe the core of the task)
- a description in the docstring
- an argument schema. 

The `@tool` decorator builds all three from your function. 

**The docstring is not a comment here; the model reads it to decide when to call the tool. Write it carefully.**

In [ ]:
# import the tool decorator from langchain's tools module
from langchain.tools import tool

#TODO: define `get_current_time` as a tool

Inspect what the decorator built:

In [ ]:
print(get_current_time.name)
print(get_current_time.description)
print(get_current_time.args)

In [ ]:
get_current_time.invoke({})

## 3. Give the tool to the model

`bind_tools` attaches tool schemas to the model as a **list** of tools.

The model does not gain the ability to run anything. It only gains the ability to ask.

In [ ]:
#TODO: bind the tool to the model

In [ ]:
from langchain.messages import HumanMessage

question = HumanMessage(content="What time is it right now?")
response = model_with_tools.invoke([question])

In [ ]:
pprint(response)

In [ ]:
response.pretty_print()

Note: `content` is empty. Instead of answering, the model produced a `tool_calls` entry naming our tool.

Remember the model proposes; it never executes.

In [ ]:
pprint(response.response_metadata)

In [ ]:
print(response.tool_calls)


Executing is our job. Run the tool with the call the model requested:

In [ ]:
call = response.tool_calls[0]
tool_result = get_current_time.invoke(call)
print(tool_result)

That produced a `ToolMessage`. Now we can combine the full conversation, including the model's tool call and the tool's result, and send it back:

In [ ]:
messages = [question, response, tool_result]

final = model_with_tools.invoke(messages)


In [ ]:
pprint(final)

In [ ]:
print(final.content)

### Interim Summary

1. The model saw the question and proposed a tool call.
2. Our code executed the tool.
3. The result went back into the conversation.
4. The model read the result and answered.

Model proposes, runtime executes, result returns. That loop is an agent.

## 4. `create_agent`: Defining the Agentic Loop

The loop so far looked like this:
 propose, execute, return, repeat until the model answers. 
 
 Because this is such a common pattern, LangChain provides a dedicated method to run the loop: `create_agent`
 
 - https://reference.langchain.com/python/langchain/agents/factory/create_agent

 Basic signature: `create_model(model, tools, system_prompt)`


In [ ]:
system_prompt = """You're a helpful assistant who answers users' questions concisely. 
                    Use the tools available when necessary."""

In [ ]:
from langchain.agents import create_agent

#TODO: define an agent with model, tool list, and system prompt


In [ ]:
result = agent.invoke({"messages": question})

for m in result["messages"]:
    m.pretty_print()

## 6. Multiple tools

Real agents have several tools and choose between them. Models are bad at date arithmetic, so we'll give our agent a second tool for that.

In [ ]:
@tool
def count_days_until_given_date(date: str) -> str:
    """Return the number of days from today until a future date given as YYYY-MM-DD."""
    target = datetime.strptime(date, "%Y-%m-%d").date()
    delta = (target - datetime.now().date()).days
    return f"{delta} days"


In [ ]:
agent = create_agent(model=model, 
                    tools=[get_current_time,count_days_until_given_date],
                    system_prompt = """You're a helpful assistant who answers users' questions concisely. 
                    Use the tools available when necessary."""
                    )


In [ ]:
question_day = HumanMessage(content="How many days until the midterm on 2026-10-30?")
response = agent.invoke({"messages":question_day })


In [ ]:
pprint(response)

The model picked the right tool and filled the argument from the question. Since executing calls is always the same steps, write a small helper:

## 6. Tools that call real APIs

**Motivation for web search:**
GPT-4.1-mini has a knowledge cutoff date of June 1, 2024.
 - https://developers.openai.com/api/docs/models/gpt-4.1-mini

 Qwen3.5's knowledge cutoff date is late 2024 to early 2025.

 Gemini 3.1 Flash-Lite has a knowledge cutoff date of January 2025.

They won't be able to answer questions about events that may have happened after the cutoff date

In [ ]:
# same agent as before
agent = create_agent(model=model, 
                    tools=[get_current_time, days_until],
                    system_prompt = """You're a helpful assistant who answers users' questions concisely. 
                    Use the tools available when necessary."""
                    )

In [ ]:
questions = [HumanMessage("Which country won the FIFA World Cup 2026"), 
            HumanMessage("Who won the Super Bowl 2026?"), 
            HumanMessage("Who is the current president of the US")]

In [ ]:
result = agent.invoke({"messages": questions[-1]})
pprint(result)


In [ ]:
pprint(result["messages"][-1].content)

In [ ]:
# loop through questions and then invoke the agent
for q in questions:
    result = agent.invoke({"messages": q})
    pprint(result["messages"][0].content)
    pprint(result["messages"][-1].content)
    print("")

### Tavily Web Search API

Let's add web search capability to our agent. We'll use Tavily as our web search API.
- https://app.tavily.com/

We'll need to provide an API key in the `.env` file, which has a `TAVILY_API_KEY` field.

#### TavilySearch
Because Tavily is specialized for agentic web search, LangChain supports it.

The `TavilySearch` method queries the Tavily Search API and gets back json.

https://reference.langchain.com/python/langchain-tavily/tavily_search/TavilySearch


In [ ]:
from langchain_tavily import TavilySearch
from typing import Dict, Any

@tool
def search_the_web(query: str) -> Dict[str, Any]:
    """Search the web for information"""
    tavily = TavilySearch(max_results=3)
    query_dict = {"query": query}
    results = tavily.invoke(query_dict)
    return results


In [ ]:
data = search_the_web.invoke({"query":"Who won the super bowl 2026"})

In [ ]:
search_docs = data.get("results", data)


In [ ]:
pprint(search_docs)

Our agent has three tools now:

In [ ]:
from langchain.agents import create_agent

agent = create_agent(model=model, 
                    tools=[get_current_time, days_until, search_the_web],
                    system_prompt = "You're a helpful assistant who answers users' questions concisely. Use the tools available when necessary."
)

In [ ]:
result = agent.invoke({"messages": questions[0]})

In [ ]:
pprint(result)

In [ ]:
for q in questions:
    result = agent.invoke({"messages": q})
    pprint(result["messages"][0].content)
    pprint(result["messages"][-1].content)
    

In [ ]:
# the response from the model
pprint(result["messages"][-1].content)

## 7. ICA: Build a weather tool

Your turn. [wttr.in](https://wttr.in) is a weather service that takes a city name directly and needs no key. One request:

```
https://wttr.in/Boston?format=j1
```

returns JSON. 

The current conditions live in `data["current_condition"][0]`, with fields including `temp_F`, `weatherDesc`, and `windspeedMiles`.

Boston or Boston,MA are acceptable. If city name matches multiple locations, it'll likely default to the one closest to your IP address.

### Your task:
Write a tool `get_weather(city)` that returns the current conditions for that city. Steps:

1. Build the URL from the city argument and fetch it with `requests.get(...).json()`.
2. Pull out temperature, conditions, and wind, and return a readable string.
3. Write the docstring so the model knows when to reach for this tool.
4. Add it to the agent and ask a question that needs it.

Skeleton:

In [ ]:
import requests
data = requests.get("https://wttr.in/Boston?format=j1").json()

In [ ]:
pprint(data["current_condition"][0])

In [ ]:
import requests

@tool
def get_weather(city: str) -> str:
    """TODO: describe what this tool does and what the argument means."""
    # TODO: fetch https://wttr.in/<city>?format=j1 with requests.get
    # TODO: parse the JSON and pull out current_condition and return this segment of the dictionary
    pass

In [ ]:
# Test your tool directly first:
# print(get_weather.invoke({"city": "Boston"}))

# Then give it to the agent:
# agent = create_agent(model=model, tools=[get_current_time, days_until, search, get_weather])
# result = agent.invoke({"messages": [HumanMessage(content="Should I bring a jacket in Boston tonight?")]})
# print(result["messages"][-1].content)